#  Agente de Análise de E-Commerce
**Text-to-SQL com Google Gemini 2.5 Flash**

Este notebook demonstra o agente que transforma perguntas em português natural em queries SQL,
executa-as no banco de dados do e-commerce e retorna análises interpretadas.

**Banco de dados:** 7 tabelas · ~100k pedidos · ~33k produtos · ~3k vendedores

---

## 1. Instalação de Dependências

In [2]:
# Execute esta célula apenas na primeira vez
# \!pip install google-generativeai pandas tabulate python-dotenv fastapi uvicorn --quiet

## 2. Configuração

Defina sua chave da API do Google Gemini abaixo.  
Obtenha gratuitamente em: [aistudio.google.com](https://aistudio.google.com/apikey)

> **Dica:** Crie um arquivo `.env` na mesma pasta com `GEMINI_API_KEY=sua_chave`  
> e o agente carregará automaticamente.

In [3]:
import os

# Opção 1: defina diretamente aqui (não faça commit com a chave\!)
# os.environ['GEMINI_API_KEY'] = 'sua_chave_aqui'

# Opção 2: carrega do arquivo .env (recomendado)
from dotenv import load_dotenv
load_dotenv()

# Verifica se a chave está configurada
if os.getenv('GEMINI_API_KEY'):
    print(' GEMINI_API_KEY configurada\!')
else:
    print(' GEMINI_API_KEY não encontrada. Configure antes de continuar.')

 GEMINI_API_KEY configurada\!


<>:12: SyntaxWarning: invalid escape sequence '\!'
<>:12: SyntaxWarning: invalid escape sequence '\!'
C:\Users\caioc\AppData\Local\Temp\ipykernel_4776\2229780835.py:12: SyntaxWarning: invalid escape sequence '\!'
  print(' GEMINI_API_KEY configurada\!')


## 3. Inicialização do Agente

In [4]:
from agent import EcommerceAgent

# Inicializa o agente apontando para o banco de dados
# Ajuste o caminho se necessário
agent = EcommerceAgent(
    db_path='files/banco.db',
    model_name='gemini-2.5-flash'  # ou 'gemini-2.5-flash-8b' (mais leve)
)

Agente de E-Commerce inicializado.
  Modelo : gemini-2.5-flash
  Banco  : files/banco.db


---
## 4.  Análise de Vendas e Receita

In [5]:
# Top 10 produtos mais vendidos
resposta = agent.perguntar('Quais são os 10 produtos mais vendidos?')
print(resposta)

Os 10 produtos mais vendidos são:

1.  **Secador de Cabelo** (categoria beleza_saude) - 1076 unidades
2.  **Toalha de Banho** (categoria cama_mesa_banho) - 919 unidades
3.  **Protetor Solar** (categoria beleza_saude) - 917 unidades
4.  **Travesseiro** (categoria cama_mesa_banho) - 839 unidades
5.  **Colar** (categoria relogios_presentes) - 804 unidades
6.  **Jogo de Lençol** (categoria cama_mesa_banho) - 791 unidades
7.  **Cobertor** (categoria cama_mesa_banho) - 758 unidades
8.  **Cadeira de Escritório** (categoria moveis_decoracao) - 607 unidades
9.  **Skate** (categoria esporte_lazer) - 595 unidades
10. **Estante de Livros Luxo** (categoria moveis_decoracao) - 580 unidades

**Observações:**

*   Produtos das categorias **beleza_saude** e **cama_mesa_banho** dominam a lista dos mais vendidos, indicando uma forte demanda por esses tipos de itens.
*   O "Secador de Cabelo" se destaca como o produto mais vendido, superando os demais por uma margem considerável.


In [6]:
# Receita total por categoria de produto
resposta = agent.perguntar('Qual é a receita total por categoria de produto? Mostre as top 10 categorias.')
print(resposta)

As 10 categorias de produtos com maior receita total são:

1.  **beleza_saude**: R$ 1.258.681,34
2.  **relogios_presentes**: R$ 1.205.005,68
3.  **cama_mesa_banho**: R$ 1.036.988,68
4.  **esporte_lazer**: R$ 988.048,97
5.  **informatica_acessorios**: R$ 911.954,32
6.  **moveis_decoracao**: R$ 729.762,49
7.  **cool_stuff**: R$ 635.290,85
8.  **utilidades_domesticas**: R$ 632.248,66
9.  **automotivo**: R$ 592.720,11
10. **ferramentas_jardim**: R$ 485.256,46

**Observações:**

*   A categoria de **beleza_saude** lidera em receita, demonstrando um alto valor de vendas, mesmo que não seja a que tem os produtos mais vendidos em quantidade, como visto na análise anterior. Isso pode indicar produtos com ticket médio mais alto.
*   **Relógios e Presentes** também se destaca com uma receita muito próxima à de beleza_saude, solidificando sua posição como uma categoria de alto valor.
*   Categorias como **cama_mesa_banho**, **esporte_lazer** e **informática_acessórios** também contribuem significa

In [7]:
# Evolução mensal de vendas
resposta = agent.perguntar('Qual foi a evolução mensal do número de pedidos e da receita total?')
print(resposta)

A evolução mensal do número de pedidos e da receita total é a seguinte:

| Mês/Ano | Total de Pedidos | Receita Total (BRL) |
|:--------|-----------------:|--------------------:|
| 2016-09 |                4 |              252.24 |
| 2016-10 |              324 |            59090.48 |
| 2016-12 |                1 |               19.62 |
| 2017-01 |              800 |           138488.04 |
| 2017-02 |             1780 |           291908.01 |
| 2017-03 |             2682 |           449863.60 |
| 2017-04 |             2404 |           417788.03 |
| 2017-05 |             3700 |           592918.82 |
| 2017-06 |             3245 |           511276.38 |
| 2017-07 |             4026 |           592382.92 |
| 2017-08 |             4331 |           674396.32 |
| 2017-09 |             4285 |           727762.45 |
| 2017-10 |             4631 |           779677.88 |
| 2017-11 |             7544 |          1194882.80 |
| 2017-12 |             5673 |           878401.48 |
| 2018-01 |             72

---
## 5.  Análise de Entrega e Logística

In [8]:
# Quantidade de pedidos por status
resposta = agent.perguntar('Qual é a quantidade de pedidos por status?')
print(resposta)

Aqui está a quantidade de pedidos por status:

| Status do Pedido   | Quantidade de Pedidos |
|:-------------------|----------------------:|
| entregue           |                 96478 |
| enviado            |                  1107 |
| cancelado          |                   625 |
| indisponível       |                   609 |
| faturado           |                   314 |
| em processamento   |                   301 |
| criado             |                     5 |
| aprovado           |                     2 |

**Observações:**

*   A vasta maioria dos pedidos (96.478) foi **entregue**, o que é um bom indicativo de eficiência na logística.
*   Existe um número razoável de pedidos **enviados** (1.107) e **cancelados** (625), que merecem atenção para identificar possíveis gargalos ou problemas.
*   Pedidos com status **indisponível** (609) também representam um volume significativo e podem indicar problemas de estoque ou com os vendedores.
*   Os status **faturado**, **em processamento*

In [9]:
# % de entregas no prazo por estado
resposta = agent.perguntar(
    'Qual é o percentual de pedidos entregues no prazo por estado do consumidor? '
    'Considere apenas pedidos com status entregue.'
)
print(resposta)

Aqui está o percentual de pedidos entregues no prazo, por estado do consumidor, considerando apenas pedidos com status "entregue":

| Estado do Consumidor | Total Entregue no Prazo | Total Pedidos Entregues | Percentual no Prazo (%) |
|:---------------------|------------------------:|------------------------:|------------------------:|
| AM                   |                     141 |                     145 |                    97.2 |
| RO                   |                     236 |                     243 |                    97.1 |
| AP                   |                      65 |                      67 |                    97.0 |
| AC                   |                      77 |                      80 |                    96.3 |
| PR                   |                    4724 |                    4923 |                    96.0 |
| SP                   |                   38674 |                   40501 |                    95.5 |
| MG                   |                   1

In [10]:
# Estados com maior atraso médio
resposta = agent.perguntar(
    'Quais são os 10 estados com maior atraso médio na entrega? '
    'Considere apenas pedidos que foram entregues fora do prazo.'
)
print(resposta)

Aqui estão os 10 estados com o maior atraso médio na entrega, considerando apenas os pedidos que foram entregues fora do prazo:

| Estado do Consumidor | Atraso Médio (dias) |
|:---------------------|--------------------:|
| AP                   |                72.5 |
| RR                   |                36.4 |
| AM                   |                30.3 |
| AC                   |                18.7 |
| SE                   |                16.2 |
| CE                   |                15.2 |
| RN                   |                14.5 |
| RJ                   |                13.5 |
| PI                   |                13.3 |
| PA                   |                12.8 |

**Observações:**

*   O estado do **Amapá (AP)** se destaca com um atraso médio significativamente alto (72.5 dias), o que é um ponto crítico a ser investigado.
*   Outros estados da região Norte, como **Roraima (RR)**, **Amazonas (AM)** e **Acre (AC)**, também aparecem com atrasos médios consideráveis (a

---
## 6. Análise de Satisfação e Avaliações

In [11]:
# Média geral de avaliação
resposta = agent.perguntar('Qual é a média geral de avaliação dos pedidos? Mostre também a distribuição por nota.')
print(resposta)

A média geral de avaliação dos pedidos é de **4.11**.

A distribuição das avaliações por nota é a seguinte:

| Nota da Avaliação | Quantidade de Avaliações |
|------------------:|-------------------------:|
|                 5 |                    55560 |
|                 4 |                    18638 |
|                 3 |                     7816 |
|                 2 |                     2920 |
|                 1 |                    10373 |

**Observações:**

*   A média geral de 4.11 é positiva, indicando que a maioria dos clientes está satisfeita com suas compras.
*   A maior parte das avaliações são notas **5** (55.560), o que reforça a satisfação geral.
*   No entanto, há um número considerável de avaliações com nota **1** (10.373), o que sugere que uma parcela de clientes teve experiências muito negativas. A análise dessas avaliações de nota 1 é crucial para identificar e corrigir problemas.
*   As avaliações com notas 2 e 3 são menos frequentes, mas também representam área

In [12]:
# Top 10 vendedores por média de avaliação
resposta = agent.perguntar(
    'Quais são os 10 vendedores com maior média de avaliação? '
    'Considere apenas vendedores com pelo menos 10 avaliações.'
)
print(resposta)

Os 10 vendedores com a maior média de avaliação (considerando apenas aqueles com pelo menos 10 avaliações) são:

| Nome do Vendedor           | Média de Avaliação |
|:---------------------------|-------------------:|
| Theo da Costa              |                 5.00 |
| Sr. Léo Andrade            |                 5.00 |
| Sr. Lucas Gabriel da Cunha |                 5.00 |
| Pedro Lucas Marques        |                 5.00 |
| Mariah Vargas              |                 5.00 |
| Maria Fernanda Peixoto     |                 5.00 |
| Luiz Otávio Abreu          |                 5.00 |
| Luiz Miguel Pacheco        |                 5.00 |
| Henrique Rios              |                 5.00 |
| Emanuel Dias               |                 5.00 |

**Observações:**

*   Todos os 10 vendedores listados alcançaram a nota máxima de 5.00, o que indica um alto nível de satisfação dos clientes com seus produtos e serviços.
*   É importante notar que esta lista considera apenas a média das ava

In [13]:
# Categorias com maior taxa de avaliação negativa
resposta = agent.perguntar(
    'Quais categorias de produto têm maior taxa de avaliação negativa (notas 1 ou 2)? '
    'Mostre as top 10 categorias com mais avaliações ruins.'
)
print(resposta)

Aqui estão as 10 categorias de produtos com maior taxa de avaliações negativas (notas 1 ou 2), considerando apenas categorias com pelo menos 10 avaliações:

| Categoria                                     | Total Avaliações Negativas | Total Avaliações | Percentual Avaliações Negativas (%) |
|:----------------------------------------------|---------------------------:|-----------------:|------------------------------------:|
| portateis_cozinha_e_preparadores_de_alimentos |                          5 |               13 |                                38.5 |
| fashion_roupa_masculina                       |                         37 |              128 |                                28.9 |
| fraldas_higiene                               |                         10 |               39 |                                25.6 |
| moveis_escritorio                             |                        382 |             1590 |                                24.0 |
| la_cuisine               

---
## 7.  Análise de Consumidores

In [14]:
# Estados com maior volume e ticket médio
resposta = agent.perguntar(
    'Quais estados têm maior volume de pedidos e maior ticket médio? '
    'Mostre os 10 principais estados com total de pedidos e valor médio por pedido.'
)
print(resposta)

Aqui estão os 10 principais estados com maior volume de pedidos, juntamente com o valor médio por pedido para cada um:

| Estado do Consumidor | Total de Pedidos | Valor Médio por Pedido (BRL) |
|:---------------------|-----------------:|-----------------------------:|
| SP                   |            41746 |                       143.69 |
| RJ                   |            12852 |                       166.85 |
| MG                   |            11635 |                       160.92 |
| RS                   |             5466 |                       162.99 |
| PR                   |             5045 |                       160.78 |
| SC                   |             3637 |                       171.32 |
| BA                   |             3380 |                       182.44 |
| DF                   |             2140 |                       165.95 |
| ES                   |             2033 |                       160.34 |
| GO                   |             2020 |            

In [15]:
# Estados com maior atraso
resposta = agent.perguntar(
    'Quais são os estados dos consumidores com maior atraso médio na entrega? '
    'Mostre os top 10 com o atraso médio em dias.'
)
print(resposta)

Os 10 estados com o maior atraso médio na entrega, considerando apenas os pedidos que foram entregues fora do prazo, são:

| Estado do Consumidor | Atraso Médio (dias) |
|:---------------------|--------------------:|
| AP                   |                72.5 |
| RR                   |                36.4 |
| AM                   |                30.3 |
| AC                   |                18.7 |
| SE                   |                16.2 |
| CE                   |                15.2 |
| RN                   |                14.5 |
| RJ                   |                13.5 |
| PI                   |                13.3 |
| PA                   |                12.8 |

**Observações:**

*   O estado do **Amapá (AP)** apresenta um atraso médio extremamente alto de 72.5 dias, o que é um ponto de preocupação significativo e sugere grandes desafios logísticos ou operacionais na região.
*   Outros estados da região Norte, como **Roraima (RR)**, **Amazonas (AM)** e **Acre (AC)**, t

---
## 8.  Análise de Vendedores e Produtos

In [16]:
# Produtos mais vendidos por estado do vendedor
resposta = agent.perguntar(
    'Quais são os 3 produtos mais vendidos em cada um dos 5 estados com maior volume de vendas?'
)
print(resposta)

Aqui estão os 3 produtos mais vendidos em cada um dos 5 estados com maior volume de vendas:

**1. São Paulo (SP)**
*   Protetor Solar: 448 unidades
*   Acessório Padrão: 435 unidades
*   Secador de Cabelo: 431 unidades

**2. Rio de Janeiro (RJ)**
*   Colar: 151 unidades
*   Toalha de Banho: 139 unidades
*   Acessório Padrão: 129 unidades

**3. Minas Gerais (MG)**
*   Secador de Cabelo: 122 unidades
*   Travesseiro: 121 unidades
*   Item Básico: 107 unidades

**4. Rio Grande do Sul (RS)**
*   Item Básico: 60 unidades
*   Cadeira de Escritório: 57 unidades
*   Toalha de Banho: 50 unidades

**5. Paraná (PR)**
*   Secador de Cabelo: 50 unidades
*   Item Básico: 45 unidades
*   Colar: 44 unidades

**Observações:**

*   **Secador de Cabelo** é um produto consistently popular, aparecendo no top 3 de SP, MG e PR.
*   **Acessório Padrão** também tem forte presença em SP e RJ.
*   **Protetor Solar** se destaca em SP.
*   **Colar** é bem vendido no RJ e PR.
*   **Toalha de Banho** aparece bem no 

In [17]:
# Top 10 vendedores por receita
resposta = agent.perguntar(
    'Quais são os 10 vendedores com maior receita total em R$? '
    'Mostre também a quantidade de pedidos de cada um.'
)
print(resposta)

Aqui estão os 10 vendedores com maior receita total em R$, juntamente com a quantidade de pedidos de cada um:

| Nome do Vendedor          | Receita Total (BRL) | Total de Pedidos |
|:--------------------------|--------------------:|-----------------:|
| Murilo Costa              |           229472.63 |             1132 |
| Ana Costa                 |           222776.05 |              358 |
| Mathias Gonçalves         |           200472.92 |             1806 |
| Isabela Santos            |           194042.03 |              585 |
| Ester Sá                  |           187923.89 |              982 |
| Matheus Vasconcelos       |           176431.87 |              336 |
| Heloisa Santos            |           160236.57 |             1314 |
| Rhavi Fernandes           |           141745.53 |             1160 |
| Srta. Stephany Gomes      |           138968.55 |              915 |
| Sra. Maria Flor Aparecida |           135171.70 |             1287 |

**Observações:**

*   **Murilo Costa

---
## 9.  Chat Interativo

Use a célula abaixo para fazer suas próprias perguntas ao agente.
O agente mantém o contexto da conversa — você pode fazer perguntas de follow-up\!

In [18]:
# Reinicia a conversa para começar com contexto limpo
agent.nova_conversa()

# Faça sua pergunta aqui\!
minha_pergunta = 'Quais são os 5 estados com maior receita total?'

resposta = agent.perguntar(minha_pergunta)
print(resposta)

Nova conversa iniciada. Historico limpo.
Os 5 estados com maior receita total são:

1.  **SP (São Paulo)**: R$ 5.998.226,96
2.  **RJ (Rio de Janeiro)**: R$ 2.144.379,69
3.  **MG (Minas Gerais)**: R$ 1.872.257,26
4.  **RS (Rio Grande do Sul)**: R$ 890.898,54
5.  **PR (Paraná)**: R$ 811.156,38

São Paulo lidera com uma receita significativamente maior que os demais estados, quase o triplo do Rio de Janeiro.


In [19]:
# Continue a conversa com uma pergunta de follow-up
followup = 'Agora me dê mais detalhes sobre o estado que apareceu em primeiro lugar.'

resposta = agent.perguntar(followup)
print(resposta)

O estado de São Paulo, que obteve a maior receita total, apresenta as seguintes categorias de produtos como as que mais contribuem para essa receita:

1.  **Cama, Mesa e Banho**: R$ 478.284,52
2.  **Beleza e Saúde**: R$ 462.305,22
3.  **Relógios e Presentes**: R$ 435.009,92
4.  **Esporte e Lazer**: R$ 386.357,01
5.  **Informática Acessórios**: R$ 350.747,88
6.  **Móveis e Decoração**: R$ 286.708,02
7.  **Utilidades Domésticas**: R$ 275.378,63
8.  **Automotivo**: R$ 214.277,27
9.  **Cool Stuff**: R$ 213.186,89
10. **Brinquedos**: R$ 185.561,83

**Observações:**

*   As categorias de "Cama, Mesa e Banho", "Beleza e Saúde" e "Relógios e Presentes" são as líderes de vendas em São Paulo, indicando uma forte demanda por produtos relacionados ao lar, bem-estar pessoal e itens de presente.
*   "Esporte e Lazer" e "Informática Acessórios" também possuem uma participação significativa, mostrando a diversidade do consumo no estado.
*   É importante notar que esses valores representam a receita to

---
## 10. ️ Chat Interativo no Terminal

Prefere uma interface de linha de comando? Execute a célula abaixo para abrir
um chat diretamente no terminal do Jupyter.

> Digite `sair` para encerrar · `novo` para reiniciar o histórico

In [ ]:
# ️ Esta célula bloqueia o notebook até você digitar 'sair'
agent.nova_conversa()
agent.chat_interativo()

Nova conversa iniciada. Historico limpo.

  Agente de Analise de E-Commerce  |  Gemini
  Digite sua pergunta sobre os dados do e-commerce.
  Comandos: 'novo' = nova conversa  |  'sair' = encerrar

